# Calculating Power-law scaling in fluctuations of information

In [1]:
import os
import numpy as np
import pandas as pd

import pyarrow.parquet as pq
from tqdm.notebook import tqdm
import warnings;warnings.filterwarnings('ignore')

In [2]:
CORPUS_NAME = 'CABNC'

In [3]:
DATA_PATH = '../data/ckpts/rosen'

OUTPUTS_PATH = 'reports'
if not os.path.exists(OUTPUTS_PATH):
    os.mkdir(OUTPUTS_PATH)

In [4]:
def grab_all_files(PATH, file_type='.parquet'):
    files = [
        [
            os.path.join(root, f) for f in files
            if f.endswith(file_type) and (not f.startswith('.'))
        ]
        for root, _, files in os.walk(PATH)
    ]
    return sum(files, [])

In [5]:
fs = [f for f in grab_all_files(DATA_PATH) if f.split('/')[-1].startswith(CORPUS_NAME)]
len(fs)

1

## Processing individual datasets

In [6]:
# formula = '{} ~ nx + ny + self'.format(COEF_VAR)

In [7]:
COEF_VAR = 'AVAR'

In [8]:
param_list = 'time_delta'

In [9]:
df_scale, df_regression, k_docs = [], [], dict()

In [10]:
for f in tqdm(fs):
    print('===][===')
    # print(f.split('/')[-1])

    table = pq.ParquetFile(f)

    df = table.read(columns=[
        'corpus', 'conversation_id', 'x_speaker', 'y_speaker',
        COEF_VAR,
        'x_turn_id', 'y_turn_id',
        # 'nx', 'ny'
    ]).to_pandas()


    df['self'] = (df['x_speaker'] == df['y_speaker'])
    df['x_turn_id'] = [int(tid.split('-')[-1]) for tid in df['x_turn_id']]
    df[param_list] = (df['y_turn_id'] - df['x_turn_id']) + 1

    df = df.loc[
        (~df[COEF_VAR].isna())
        & (~df[param_list].isna())
    ]

    for corpus in tqdm(df['corpus'].unique()):

        if corpus not in k_docs.keys():
            k_docs[corpus] = {
                True: 0,
                False: 0
            }

        sub = df.loc[df['corpus'].isin([corpus])]

        groups = sub.groupby(by=['conversation_id', 'x_speaker', 'self'])
        for labels, dfi in tqdm(groups):

            if len(dfi):
                k_docs[corpus][labels[2]] += 1

                b,a = np.polyfit(np.log(dfi[param_list].values), np.log(dfi['AVAR'].values + 1e-9), 1)

                df_regression += [
                    {
                        'corpus': corpus,
                        'cid': labels[0],
                        'speaker': labels[1],
                        'self': labels[2],
                        'a': np.exp(a),
                        'b': b,
                        'k': len(dfi)
                    }
                ]

df_regression = pd.DataFrame(df_regression)

  0%|          | 0/1 [00:00<?, ?it/s]

===][===


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/10812 [00:00<?, ?it/s]

In [ ]:
# df_regression.isna().sum()

In [ ]:
# df_regression = df_regression.loc[
#     (df_regression.isna().astype(int).sum(axis=1) == 0)
#     & (np.isinf(df_regression).astype(int).sum(axis=1) == 0)
# ]

In [11]:
if not os.path.exists(os.path.join(OUTPUTS_PATH,'all.csv')):
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        encoding='utf-8'
    )

else:
    df_regression.to_csv(
        os.path.join(OUTPUTS_PATH,'all.csv'),
        index=False,
        header=False,
        encoding='utf-8',
        mode='a'
    )

## Corpus level analysis of results

In [12]:
split_by = ['corpus', 'self']

In [13]:
df0 = df_regression[split_by+['b']].groupby(by=split_by).agg('mean').reset_index()
df0['std'] = df_regression[split_by+['b']].groupby(by=split_by).agg('std').reset_index()['b'].values

df0['k'] = [k_docs[corpus][s] for corpus, s in df0[['corpus', 'self']].values]

df0['se'] = df0['std'] / np.sqrt(df0['k'])

In [14]:
df0['t'] = df0['b'] / df0['se']

In [15]:
df0.head(n=20)

,corpus,self,b,std,k,se,t
0,CABNC,False,-0.441987,1.315385,5761,0.017330,-25.503822
1,CABNC,True,-0.520260,1.475097,5051,0.020755,-25.066218


In [16]:
df0.to_csv(
    os.path.join(OUTPUTS_PATH, CORPUS_NAME+'.csv'),
    index=False, encoding='utf-8'
)